# Previsão de séries temporais com Prophet

# 1. Configurações

Vamos fazer a importação de bibliotecas essenciais para análise de dados, visualização e modelagem:

- `datetime, numpy, pandas`: Para manipulação de datas, operações numéricas e estruturação de dados.
- `seaborn, matplotlib`: Para criar gráficos e visualizações.
- `Prophet`: Para modelar séries temporais.
- `sklearn.metrics`: Para calcular métricas de erro de previsão.
- `plt.style.use('fivethirtyeight')`: Define um estilo de gráfico que melhora a estética das visualizações.

Essas importações preparam o ambiente para análises preditivas e visualizações robustas.

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar
plt.style.use('fivethirtyeight')

Este código usa o gdown para baixar um arquivo CSV do Google Drive:

- `URL`: O arquivo localizado na URL especificada será baixado.
- `-O ./PJME_hourly.csv`: O arquivo será salvo localmente com o nome PJME_hourly.csv.

Isso facilita o acesso direto aos dados sem a necessidade de downloads manuais.

In [ ]:
!gdown 1_Szq48LX9-yUsS5uenoi4kI7qwyAOeaw -O ./PJME_hourly.csv

# 2. Leitura e exploração dos dados

Agora, vamos carregar o arquivo CSV baixado em um DataFrame do pandas:

- `pd.read_csv`: Lê o arquivo PJME_hourly.csv.
- `index_col=[0]`: Define a primeira coluna como índice.
- `parse_dates=[0]`: Converte a coluna de índice em formato de data.

Em seguida, o DataFrame df é exibido para uma visualização inicial dos dados.

In [ ]:
df = pd.read_csv("./PJME_hourly.csv", index_col=[0], parse_dates=[0])
df

Agora, vamos gerar um gráfico de linhas do DataFrame df:

- `df.plot`: Cria o gráfico, onde o eixo x será o índice (datas) e o eixo y os valores.
- `figsize=(15,5)`: Define o tamanho da figura.
- `style='-'`: Usa uma linha contínua no gráfico.
- `linewidth=1`: Define a espessura da linha.
- `title='PJM East'`: Adiciona um título ao gráfico.

Em seguida, o gráfico é exibido com `plt.show()`.

In [ ]:
df.plot(figsize=(15,5), style='-', linewidth=1, title='PJM East')
plt.show()

Vamos criar um gráfico de linhas para as primeiras 100 linhas do DataFrame df:

- `df.iloc[:100].plot`: Plota as primeiras 100 entradas.
- `figsize=(15,5)`: Define o tamanho da figura.
- `style='-'`: Usa uma linha contínua.
- `marker='o'`: Adiciona marcadores em cada ponto de dados.
- `linewidth=1`: Define a espessura da linha.
- `title='PJM East'`: Adiciona um título ao gráfico.

Em seguida, o gráfico é exibido com `plt.show()`.

In [ ]:
df.iloc[:100].plot(figsize=(15,5), style='-', marker='o', linewidth=1, title='PJM East')
plt.show()

# 3. Divisão treino-teste

Agora, vamos dividir o DataFrame df em conjuntos de treinamento e teste com base na data split_date:

- `split_date = '01-Jan-2015'`: Define a data de divisão.
- `df_train = df.loc[df.index <= split_date].copy()`: Cria o conjunto de treinamento com dados até a data especificada.
- `df_test = df.loc[df.index > split_date].copy()`: Cria o conjunto de teste com dados após a data especificada.

Esses conjuntos serão usados para treinar e avaliar modelos.

In [ ]:
split_date = '01-Jan-2015'

df_train = df.loc[df.index <= split_date].copy()

df_test = df.loc[df.index > split_date].copy()

Vamos preparar e visualizar os conjuntos de dados de treinamento e teste:

- `df_plot = df_train.rename(columns={'PJME_MW': 'TRAINING SET'})`: Renomeia a coluna no conjunto de treinamento para TRAINING SET.
- `df_plot = df_plot.join(df_test.rename(columns={'PJME_MW': 'TEST SET'}), how='outer')`: Adiciona o conjunto de teste ao DataFrame, renomeando a coluna para TEST SET e mantendo todos os dados com a junção externa.

Finalmente, o gráfico é gerado para comparar os conjuntos de treinamento e teste com `df_plot.plot` e exibido com `plt.show()`.

In [ ]:
df_plot = df_train.rename(columns={'PJME_MW': 'TRAINING SET'})
df_plot = df_plot.join(df_test.rename(columns={'PJME_MW': 'TEST SET'}), how='outer')
df_plot.plot(figsize=(15,5), title='PJM East', style='-', linewidth=1)
plt.show()

Agora, vamos preparar os conjuntos de dados para o modelo Prophet:

- `df_train = df_train.reset_index().rename(columns={'Datetime':'ds', 'PJME_MW':'y'})`: Reseta o índice do conjunto de treinamento e renomeia as colunas para ds (data) e y (valores).
- `df_test = df_test.reset_index().rename(columns={'Datetime':'ds', 'PJME_MW':'y'})`: Faz o mesmo para o conjunto de teste.

Essas etapas ajustam os nomes das colunas para os padrões esperados pelo Prophet.

In [ ]:
df_train = df_train.reset_index().rename(columns={'Datetime':'ds', 'PJME_MW':'y'})
df_test = df_test.reset_index().rename(columns={'Datetime':'ds', 'PJME_MW':'y'})

# 4. Modelo Prophet e ajuste dos dados

Vamos ajustar o modelo Prophet com o conjunto de dados de treinamento:

- `model = Prophet()`: Cria uma instância do modelo Prophet.
- `model.fit(df_train)`: Ajusta o modelo aos dados de treinamento, com as colunas renomeadas para ds (data) e y (valores).

Essas etapas preparam o modelo para fazer previsões com base nos dados históricos.

In [ ]:
model = Prophet()
model.fit(df_train)

# 5. Previsão do conjunto de teste

Vamos fazer previsões com o modelo Prophet e exibir os resultados:

- `df_forecast = model.predict(df=df_test)`: Gera previsões para o conjunto de teste usando o modelo ajustado.
- `df_forecast`: Exibe o DataFrame com as previsões, que incluirá colunas como ds (data), yhat (previsão) e intervalos de confiança.

Essas previsões serão usadas para avaliar o desempenho do modelo.

In [ ]:
df_forecast = model.predict(df=df_test)
df_forecast

Vamos visualizar as previsões feitas pelo modelo Prophet:

- `f, ax = plt.subplots(1)`: Cria uma figura e um eixo para o gráfico.
- `f.set_figheight(5) e f.set_figwidth(15)`: Define a altura e largura da figura.
- `fig = model.plot(df_forecast, ax=ax)`: Plota as previsões usando o método plot do Prophet e o eixo criado.

O gráfico exibido com `plt.show()` incluirá as previsões, a série temporal original e os intervalos de confiança.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
fig = model.plot(df_forecast, ax=ax)
plt.show()

Vamos visualizar os componentes das previsões do modelo Prophet:

- `fig = model.plot_components(df_forecast)`: Plota os componentes do modelo Prophet, como tendência, sazonalidade diária e sazonalidade anual, usando o DataFrame df_forecast.

O gráfico exibido com `plt.show()` permitirá analisar os diferentes componentes que influenciam as previsões.

In [ ]:
fig = model.plot_components(df_forecast)
plt.show()

Vamos visualizar as previsões juntamente com os valores reais:

- `f, ax = plt.subplots(1)`: Cria uma figura e um eixo para o gráfico.
- `f.set_figheight(5) e f.set_figwidth(15)`: Define a altura e largura da figura.
- `ax.scatter(df_test['ds'], df_test['y'], color='r')`: Adiciona os valores reais do conjunto de teste ao gráfico como pontos vermelhos.
- `fig = model.plot(df_forecast, ax=ax)`: Plota as previsões do modelo Prophet no mesmo eixo.

O gráfico exibido com `plt.show()` permite comparar visualmente as previsões com os dados reais.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
ax.scatter(df_test['ds'], df_test['y'], color='r', )
fig = model.plot(df_forecast, ax=ax)

Vamos criar um gráfico comparando previsões e valores reais para janeiro de 2015:

- `f, ax = plt.subplots(1)`: Cria uma figura e um eixo para o gráfico.
- `f.set_figheight(5) e f.set_figwidth(15)`: Define a altura e largura da figura.
- `df_test.set_index("ds").plot(figsize=(15,5), style='-', color='r', ax=ax)`: Plota os valores reais do conjunto de teste como uma linha vermelha.
- `fig = model.plot(df_forecast, ax=ax)`: Adiciona as previsões ao gráfico.
- `ax.set_xbound(lower=datetime.strptime('2015-01-01', '%Y-%m-%d'), upper=datetime.strptime('2015-02-01', '%Y-%m-%d'))`: Define o intervalo do eixo x para janeiro de 2015.
- `ax.set_ylim(0, 60000)`: Define o intervalo do eixo y.
- `plot = plt.suptitle('January 2015 Forecast vs Actuals')`: Adiciona um título ao gráfico.

O gráfico exibido com `plt.show()` permite comparar as previsões do modelo com os dados reais para o mês de janeiro de 2015.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
df_test.set_index("ds").plot(figsize=(15,5), style='-', color='r', ax=ax)
fig = model.plot(df_forecast, ax=ax)
ax.set_xbound(
    lower=datetime.strptime('2015-01-01', '%Y-%m-%d'),
    upper=datetime.strptime('2015-02-01', '%Y-%m-%d')
)
ax.set_ylim(0, 60000)
plot = plt.suptitle('January 2015 Forecast vs Actuals')

Vamos criar um gráfico comparando as previsões e os valores reais para a primeira semana de janeiro de 2015:

- `f, ax = plt.subplots(1)`: Cria uma figura e um eixo para o gráfico.
- `f.set_figheight(5) e f.set_figwidth(15)`: Define a altura e largura da figura.
- `df_test.set_index("ds").plot(figsize=(15,5), style='-', color='r', ax=ax)`: Plota os valores reais do conjunto de teste como uma linha vermelha.
- `fig = model.plot(df_forecast, ax=ax)`: Adiciona as previsões ao gráfico.
- `ax.set_xbound(lower=datetime.strptime('2015-01-01', '%Y-%m-%d'), upper=datetime.strptime('2015-01-08', '%Y-%m-%d'))`: Define o intervalo do eixo x para a primeira semana de janeiro de 2015.
- `ax.set_ylim(0, 60000)`: Define o intervalo do eixo y.
- `plot = plt.suptitle('First Week of January Forecast vs Actuals')`: Adiciona um título ao gráfico.

O gráfico exibido com `plt.show()` permite comparar as previsões com os dados reais para a primeira semana de janeiro de 2015.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
df_test.set_index("ds").plot(figsize=(15,5), style='-', color='r', ax=ax)
fig = model.plot(df_forecast, ax=ax)
ax.set_xbound(
    lower=datetime.strptime('2015-01-01', '%Y-%m-%d'),
    upper=datetime.strptime('2015-01-08', '%Y-%m-%d')
)
ax.set_ylim(0, 60000)
plot = plt.suptitle('First Week of January Forecast vs Actuals')

# 6. Métricas de avaliação do primeiro modelo

Vamos calcular as métricas de erro para avaliar a precisão das previsões:

- `mean_squared_error()`: Calcula o erro quadrático médio (MSE) entre os valores reais e as previsões.
- `mean_absolute_error()`: Calcula o erro absoluto médio (MAE) entre os valores reais e as previsões.
- `mean_absolute_percentage_error()`: Calcula o erro percentual absoluto médio (MAPE) entre os valores reais e as previsões.

Essas métricas ajudam a avaliar a qualidade das previsões feitas pelo modelo Prophet.

In [ ]:
mean_squared_error(
    y_true=df_test['y'],
    y_pred=df_forecast['yhat']
)

In [ ]:
mean_absolute_error(
    y_true=df_test['y'],
    y_pred=df_forecast['yhat']
)

In [ ]:
mean_absolute_percentage_error(
    y_true=df_test['y'],
    y_pred=df_forecast['yhat']
)

# 7. Adicionando feriados

Vamos identificar os feriados dentro dos períodos de treinamento e teste:

- `train_holidays = cal.holidays(start=df_train.index.min(), end=df_train.index.max())`: Obtém os feriados que ocorrem durante o período do conjunto de treinamento.
- `test_holidays = cal.holidays(start=df_test.index.min(), end=df_test.index.max())`: Obtém os feriados que ocorrem durante o período do conjunto de teste.

Esses feriados podem ser usados para ajustar o modelo Prophet e melhorar a previsão ao considerar eventos especiais.

In [ ]:
cal = calendar()

train_holidays = cal.holidays(
    start=df_train.index.min(),
    end=df_train.index.max()
)

test_holidays = cal.holidays(
    start=df_test.index.min(),
    end=df_test.index.max()
)

Vamos preparar um DataFrame para os feriados:

- `df['date'] = df.index.date`: Adiciona uma coluna com as datas (sem hora).
- `df['is_holiday'] = df.date.isin([d.date() for d in cal.holidays()])`: Marca os dias que são feriados.
- `holiday_df = df.loc[df['is_holiday']]`: Filtra o DataFrame para manter apenas os feriados.
- `holiday_df = holiday_df.reset_index().rename(columns={'Datetime':'ds'})`: Reseta o índice e renomeia a coluna de data.
- `holiday_df['holiday'] = 'USFederalHoliday'`: Adiciona uma coluna identificando o tipo de feriado.
- `holiday_df = holiday_df.drop(['PJME_MW','date','is_holiday'], axis=1)`: Remove colunas desnecessárias.
- `holiday_df['ds'] = pd.to_datetime(holiday_df['ds'])`: Converte a coluna de datas para o formato datetime.

O DataFrame `holiday_df` contém as datas dos feriados, preparado para ser integrado ao modelo Prophet.

In [ ]:
df['date'] = df.index.date
df['is_holiday'] = df.date.isin([d.date() for d in cal.holidays()])
holiday_df = df.loc[df['is_holiday']]
holiday_df = holiday_df.reset_index().rename(columns={'Datetime':'ds'})
holiday_df['holiday'] = 'USFederalHoliday'
holiday_df = holiday_df.drop(['PJME_MW','date','is_holiday'], axis=1)
holiday_df['ds'] = pd.to_datetime(holiday_df['ds'])
holiday_df


Vamos ajustar o modelo Prophet para incluir os feriados. Essa abordagem pode melhorar as previsões ao levar em conta os efeitos dos feriados na série temporal.

In [ ]:
model_with_holidays = Prophet(holidays=holiday_df)
model_with_holidays.fit(df_train)

Vamos fazer previsões com o modelo Prophet que inclui feriados. O DataFrame `df_forecast_with_holidays` contém as previsões ajustadas para feriados.

In [ ]:
df_forecast_with_holidays = model_with_holidays.predict(df=df_test.reset_index())
df_forecast_with_holidays = df_forecast_with_holidays.rename(columns={'Datetime':'ds'})
df_forecast_with_holidays

Vamos visualizar os componentes do modelo Prophet ajustado para feriados, plotando os componentes do modelo, incluindo tendências, sazonalidades e o impacto dos feriados.

O gráfico exibido ajudará a entender como os feriados e outros componentes afetam as previsões.

In [ ]:
fig2 = model_with_holidays.plot_components(df_forecast_with_holidays)

Vamos calcular as métricas de erro para avaliar as previsões ajustadas com feriados. Essas métricas ajudam a comparar a performance do modelo ajustado com feriados em relação ao modelo anterior.

In [ ]:
mean_squared_error(
    y_true=df_test['y'],
    y_pred=df_forecast_with_holidays['yhat']
)

In [ ]:
mean_absolute_error(
    y_true=df_test['y'],
    y_pred=df_forecast_with_holidays['yhat']
)

In [ ]:
mean_absolute_percentage_error(
    y_true=df_test['y'],
    y_pred=df_forecast_with_holidays['yhat']
)

Vamos criar um gráfico comparando as previsões e os valores reais para a semana do dia 4 de julho.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
ax.scatter(df_test["ds"], df_test['y'], color='r')
fig = model.plot(df_forecast, ax=ax)
ax.set_xbound(
    lower=datetime.strptime('2015-07-01', '%Y-%m-%d'),
    upper=datetime.strptime('2015-07-07', '%Y-%m-%d')
)
ax.set_ylim(0, 60000)
plot = plt.suptitle('Week of July 4th Forecast vs Actuals non-Holiday Model')

Agora podemos comparar com o mesmo período utilizando o modelo de feriados.

In [ ]:
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
ax.scatter(df_test["ds"], df_test['y'], color='r')
fig = model.plot(df_forecast_with_holidays, ax=ax)
ax.set_xbound(
    lower=datetime.strptime('2015-07-01', '%Y-%m-%d'),
    upper=datetime.strptime('2015-07-07', '%Y-%m-%d')
)
ax.set_ylim(0, 60000)
plot = plt.suptitle('Week of July 4th Forecast vs Actuals Holiday Model')

As métricas de avaliação podem ser calculadas para os dados de 4 de julho.

In [ ]:
jul4_test = df_test.query('ds >= 20160407 and ds < 20160408')
jul4_pred = df_forecast.query('ds >= 20160407 and ds < 20160408')
jul4_pred_holiday = df_forecast_with_holidays.query('ds >= 20160407 and ds < 20160408')

In [ ]:
mean_squared_error(
    y_true=jul4_test['y'],
    y_pred=jul4_pred['yhat']
)

In [ ]:
mean_squared_error(
    y_true=jul4_test['y'],
    y_pred=jul4_pred_holiday['yhat']
)